In [1]:
import cudf as pd, numpy as np, matplotlib.pyplot as plt, json, os

# Microdatos del Censo 2024

## Introducción

La liberación del los datos del censo 2024 de vivienda y personas es fundamental para la investigación, toma de decisiones y el diseño de políticas públicas. 

Dado que estos contienen información __sensible__ (definidas según la ley 21.719 sobre proyección de datos personales), el INE debe aplicar distintos métodos para proteger la identidad de las personas y asegurar el __secreto estadístico__ (ley 17.374 sobre la creación del INE). 

Este mandato provoca un desafío natural, proteger la privacidad de las personas, al mismo tiempo que se mantiene la útilidad estadística de los datos. Entre los cambios más importantes para la liberación de los últimos microdatos, el INE decidió que la máxima desagregación geográfica fuera __comuna__. Para niveles geográficos más pequeños, se liberaron los conteos de ciertas características. 

Esto genera un problema y plantea una pregunta, primero __no existen datos desagregados a nivel de manzana__ y segundo, *¿Cómo podemos generar datos sintéticos desagregados a nivel de manzana?*. El objetivo de este trabajo es generar una base de microdatos a nivel de manzana.

_Secreto estadístico: Obligación legal de mantener en absoluta reserva la información individual entregada a los organismos que conforman el Sistema Estadístico Nacional._    

## Análisis Exploratorio

In [2]:
df_personas  = pd.read_csv(os.path.join("databases","personas_censo2024.csv"), sep=";")
df_hogares   = pd.read_csv(os.path.join("databases","hogares_censo2024.csv"), sep=";")
df_viviendas = pd.read_csv(os.path.join("databases","viviendas_censo2024.csv"), sep=";")
df_manzanas  = pd.read_csv(os.path.join("databases","Base_manzana_entidad_CPV24.csv"), sep=";")

### Microdatos nivel comuna

In [3]:
display(df_personas.head())
len(df_personas)

,id_vivienda,id_hogar,id_persona,region,provincia,comuna,comuna_bajo_umbral,area,tipo_operativo,sexo,...,p45_medio_transporte,p46a_tot_hijs_nac,p46b_hijas_nac,p46c_hijos_nac,p47a_tot_hijs_sobrev,p47b_hijas_sobrev,p47c_hijos_sobrev,p48_anio_nac_uh,p48_mes_nac_uh,div_genero
0,1,1,1,5,58,5802,2,1,2,2,...,<NA>,3,2,1,3,2,1,1978,7,2
1,1,1,2,5,58,5802,2,1,2,1,...,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2
2,1,1,3,5,58,5802,2,1,2,2,...,3,1,1,0,1,1,0,2015,9,2
3,1,1,4,5,58,5802,2,1,2,2,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,2,1,1,4,43,4303,2,2,2,1,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-66


18480432

In [4]:
df_personas.columns

Index(['id_vivienda', 'id_hogar', 'id_persona', 'region', 'provincia',
       'comuna', 'comuna_bajo_umbral', 'area', 'tipo_operativo', 'sexo',
       'edad', 'edad_quinquenal', 'parentesco', 'p23_est_civil',
       'p24_lug_resid5', 'p24_lug_resid5_esp', 'p25_lug_nacimiento',
       'p25_lug_nacimiento_rec', 'p25_lug_nacimiento_esp',
       'p26_llegada_periodo', 'p27_nacionalidad', 'p27_nacionalidad_esp',
       'p27_nacionalidad_rec', 'p28_autoid_pueblo', 'p28_pueblo_pert',
       'p29_afrodescendencia_rec', 'p29_afrodescendencia',
       'p30_lengua_indigena', 'p30_lengua_indigena_rec', 'p31_religion',
       'p31_religion_rec', 'p32a_dificultad_ver', 'p32b_dificultad_oir',
       'p32c_dificultad_mover', 'p32d_dificultad_cogni',
       'p32e_dificultad_cuidado', 'p32f_dificultad_comunic', 'discapacidad',
       'p33_edu_asiste', 'asistencia_parv', 'asistencia_basica',
       'asistencia_media', 'asistencia_superior', 'p37_alfabet', 'escolaridad',
       'cine11', 'sit_fuerza_traba

Cada fila corresponde a una persona censada, estos datos contienen información como por ejemplo:
+ Comuna 
+ Sexo
+ Edad (en años cumplidos)
+ Medio de transporte más usado
+ Pertenencia a pueblo originario
+ Religión

In [5]:
display(df_viviendas.head())
len(df_viviendas)

,id_vivienda,region,provincia,comuna,comuna_bajo_umbral,area,tipo_operativo,cant_hog,cant_per,p2_tipo_vivienda,...,p5_num_dormitorios,p6_fuente_agua,p7_distrib_agua,p8_serv_hig,p9_fuente_elect,p10_basura,p11a_num_personas,p11b_comparte_gasto,p11c_num_hogar,indice_hacinamiento
0,1,5,58,5802,2,1,2,1,4,1,...,2,1,1,1,1,1,4,1,<NA>,1
1,2,4,43,4303,2,2,2,1,3,1,...,2,1,1,3,1,1,3,1,<NA>,1
2,3,11,112,11202,1,1,2,1,3,1,...,3,1,1,1,1,1,3,1,<NA>,1
3,4,1,11,1101,2,1,2,1,3,3,...,2,1,1,1,1,1,3,1,<NA>,1
4,5,8,83,8301,2,1,2,1,1,1,...,1,1,1,1,1,1,1,<NA>,<NA>,1


7664466

In [6]:
df_viviendas.columns

Index(['id_vivienda', 'region', 'provincia', 'comuna', 'comuna_bajo_umbral',
       'area', 'tipo_operativo', 'cant_hog', 'cant_per', 'p2_tipo_vivienda',
       'p3a_estado_ocupacion', 'p3b_estado_ocupacion', 'p4a_mat_paredes',
       'p4b_mat_techo', 'p4c_mat_piso', 'p5_num_dormitorios', 'p6_fuente_agua',
       'p7_distrib_agua', 'p8_serv_hig', 'p9_fuente_elect', 'p10_basura',
       'p11a_num_personas', 'p11b_comparte_gasto', 'p11c_num_hogar',
       'indice_hacinamiento'],
      dtype='object')

Contiene información sobre la vivienda de las personas, enfocandose en los aspectos físicos de esta, por ejemplo:
+ Tipo de vivienda
+ Material de las paredes 
+ Material del piso
+ Número de dormitorios

In [7]:
display(df_hogares.head())
len(df_hogares)

,id_vivienda,id_hogar,region,provincia,comuna,comuna_bajo_umbral,area,tipo_operativo,p12_tenencia_viv,p13_comb_cocina,p14_comb_calefaccion,p15a_serv_tel_movil,p15b_serv_compu,p15c_serv_tablet,p15d_serv_internet_fija,p15e_serv_internet_movil,p15f_serv_internet_satelital,tipologia_hogar
0,1,1,5,58,5802,2,1,2,4,1,8,1,2,2,2,1,2,5
1,2,1,4,43,4303,2,2,2,9,1,3,1,2,2,2,1,2,7
2,3,1,11,112,11202,1,1,2,3,1,3,1,2,2,2,1,2,4
3,4,1,1,11,1101,2,1,2,3,1,1,1,1,1,1,1,2,4
4,5,1,8,83,8301,2,1,2,1,1,2,1,2,1,1,1,2,1


6622597

In [8]:
df_hogares.columns

Index(['id_vivienda', 'id_hogar', 'region', 'provincia', 'comuna',
       'comuna_bajo_umbral', 'area', 'tipo_operativo', 'p12_tenencia_viv',
       'p13_comb_cocina', 'p14_comb_calefaccion', 'p15a_serv_tel_movil',
       'p15b_serv_compu', 'p15c_serv_tablet', 'p15d_serv_internet_fija',
       'p15e_serv_internet_movil', 'p15f_serv_internet_satelital',
       'tipologia_hogar'],
      dtype='object')

Contiene información respecto al grupo social que vive en esa vivienda, enfocandose en la dínamica económica común. Dentro de las variables que miden tenemos: 
+ Tenencia (si es pagada, pagandose, arrendada, etc.)
+ Con que funciona la cocina
+ Principal fuente de calefacción
+ Tipología del hogar 
+ Servicio de internet

Es importante reecalcar que mediante las variables ```id_hogar``` y ```id_vivienda``` nos permite enlazar a cada persona con un hogar y vivienda.

### Datos nivel manzana - entidad

In [9]:
display(df_manzanas.head())
len(df_manzanas)

,CONTENEDOR_COMUNAL,COD_REGION,REGION,PROVINCIA,CUT,COMUNA,AREA_C,MANZENT,DISTRITO,COD_DISTRITO,...,n_fuente_elect_diesel,n_fuente_elect_solar,n_fuente_elect_eolica,n_fuente_elect_otro,n_fuente_elect_no_tiene,n_basura_servicios,n_basura_entierra,n_basura_eriazo,n_basura_rio,n_basura_otro
0,0,1,TARAPACÁ,IQUIQUE,1101,IQUIQUE,1,1101011001007,PUERTO,1,...,0,0,0,0,0,4,0,0,0,0
1,0,1,TARAPACÁ,IQUIQUE,1101,IQUIQUE,1,1101011001010,PUERTO,1,...,0,0,0,0,0,11,0,0,0,0
2,0,1,TARAPACÁ,IQUIQUE,1101,IQUIQUE,1,1101011001012,PUERTO,1,...,0,0,0,0,0,5,0,0,0,0
3,0,1,TARAPACÁ,IQUIQUE,1101,IQUIQUE,1,1101011001013,PUERTO,1,...,0,0,0,0,2,20,0,0,0,0
4,0,1,TARAPACÁ,IQUIQUE,1101,IQUIQUE,1,1101011001014,PUERTO,1,...,0,0,0,0,0,0,0,0,0,0


197032

In [10]:
list(df_manzanas.columns)

['CONTENEDOR_COMUNAL',
 'COD_REGION',
 'REGION',
 'PROVINCIA',
 'CUT',
 'COMUNA',
 'AREA_C',
 'MANZENT',
 'DISTRITO',
 'COD_DISTRITO',
 'COD_LOCALIDAD',
 'COD_ZONA',
 'LOCALIDAD',
 'COD_ENTIDAD',
 'COD_MANZANA',
 'ENTIDAD',
 'TIPO_MZ',
 'COD_CATEGORIA',
 'CATEGORIA',
 'ID_ENTIDAD',
 'ID_LOCALIDAD',
 'ID_DISTRITO',
 'ID_ZONA',
 'n_per',
 'n_hombres',
 'n_mujeres',
 'n_edad_0_5',
 'n_edad_6_13',
 'n_edad_14_17',
 'n_edad_18_24',
 'n_edad_25_44',
 'n_edad_45_59',
 'n_edad_60_mas',
 'prom_edad',
 'n_inmigrantes',
 'n_nacionalidad',
 'n_pueblos_orig',
 'n_afrodescendencia',
 'n_lengua_indigena',
 'n_religion',
 'n_dificultad_ver',
 'n_dificultad_oir',
 'n_dificultad_mover',
 'n_dificultad_cogni',
 'n_dificultad_cuidado',
 'n_dificultad_comunic',
 'n_discapacidad',
 'n_estcivcon_casado',
 'n_estcivcon_conviviente',
 'n_estcivcon_conv_civil',
 'n_estcivcon_anul_sep_div',
 'n_estcivcon_viudo',
 'n_estcivcon_soltero',
 'prom_escolaridad18',
 'n_asistencia_parv',
 'n_asistencia_basica',
 'n_asis

Contiene algunos estadísticos agregados sobre los 3 tópicos vistos arriba, personas, vivienda y hogar, alguno de estos conteos son:
+ Número de hombres
+ Número de Mujeres
+ Número de personas por edad quinquenal
+ Promedio de edad
+ Número de personas de pueblos originarios
+ Número de los estados civiles
+ Número de hogares unipersonales 
+ Número de viviendas con acceso a agua por red pública
+ Etc.

## Estructura 

Los datos tienen una estructura jerárquica de árbol. A nivel macro (comuna) conocemos la distribución conjunta real de la población, pero a nivel micro (manzana) solo disponemos de conteos marginales. Esto impone restricciones lógicas, como por ejemplo que la agregación de las distribuciones marginales, debemos reconstruir la distribución conjunta comunal. 

Nos gustaría hacer asignación espacial con garantías de privacidad para desagregar los datos a nivel de manzana, es decir, con cierto mecanismo de privacidad, nos gustaría a cada fila del dataset de personas, vivienda y hogares asignarle una manzana.

# El Problema de Transporte Óptimo Entrópico

Supongamos que nos gustaría estimar la distribución de los hombres por edad en una **comuna** que tiene **3 manzanas** y 1000 hombres.

De los datos que libera el INE tenemos:
+ Nivel macro (Comuna): Distribución por edad 
    - Jóvenes (0-17): 200
    - Adultos (18-64): 600
    - Mayores (65+): 200
+ Nivel micro (Manzana): Distribución de hombres
    - Manzana A: 500 hombres
    - Manzana B: 400 hombres
    - Manzana C: 100 hombres

Nuestro objetivo es descubrir como se cruza esta información, si lo tabulamos, nos encontramos con una matriz vacía:
| Edad / Manzana | A | B | C | Comuna |
| :-------------- | :-: | :-: | :-: | :------: |
| Jóvenes | ? | ? | ? | 200 |
| Adultos | ? | ? | ? | 600 |
| Mayores | ? | ? | ? | 200 |
| Manzana | 500 | 400 | 100 | 1000

Ahora, nuestro problema es encontrar valores para la matriz, ¿cómo lo hacemos?


## Transporte Óptimo Entrópico

Nuestro problema se vuelve a la creación de una matriz que respete las restricciones de las distribuciones marginales. Es decir, buscamos $P$ que minimice el "costo" de asignación espacial, regularizando por la entropía para suavizar la distribución.

Sea $\mu,\nu$ dos distribuciones de probabilidad y sea $C$ una matriz de costos (que tanto me cuesta mover masa de $\mu$ a $\nu$), buscamos una distribución $P$ en los couplings que minimice esta carga, es decir.

\begin{align*}
\min_{P} \quad&\sum_{i,j}P_{ij}C_{ij} - \epsilon H(P)\\ 
s.a \quad&P 1 = \mu \\ 
&P^T1 = \nu
\end{align*}
con $H(P) = -\sum_{ij}P_{ij}\log P_{ij}$.

En nuestro ejemplo:
+ $P$-Plan de transporte: la matriz vacía que queremos rellenar. Nos dirá cuántos jóvenes, adultos y mayores viven en las manzanas A, B y C.
+ $\mu$-Distribución de origen: El vector de edades de la comuna. Restricción de filas.
+ $\nu$-Distribución de destino: El vector de población del INE. Restricción de las columnas.
+ $C$-Matriz de costo: Es la lógica territorial, debe ser construida mediante información externa y por nosotros, hay que pensarlo como una penalización. En nuestro ejemplo, una manzana con mayoría de departamentos de 1-2 dormitorios podemos pensar que es menos probable que hayan jóvenes.
+ $\epsilon$-Parámetro de entropía: "Ruido", un valor mayor de este parámetro difumina más a los hombres entre manzanas.


## Pros y Contras de este modelo

+ Gracias a la regularización entrópica, el problema cuenta con solución analítica y se puede resolver de manera eficiente.
+ El problema es escalable y paralelizable.
+ No es claro que tanta privacidad entrega el parámetro $\epsilon$.
+ La matriz de costo $C$ es flexible, puede ser un pro y un contra a la vez.
+ El modelo produce distribuciones de probabilidad, por ejemplo, la solución nos puede entregar en una manzana 4.3 jóvenes y 2.7 adultos, nosotros necesitamos enteros. De todas formas, podemos muestrear con esta distribución.